# Install packages

In [141]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

Download minute-level stock data

In [143]:
ticker = "NVDA"

df = yf.download(
    ticker,
    period="7d",
    interval="1m"
)

df.tail()

C:\Users\Alexx\AppData\Local\Temp\ipykernel_22980\2999303205.py:3: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(
[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,NVDA,NVDA,NVDA,NVDA,NVDA
Datetime,,,,,
2026-07-09 19:55:00+00:00,203.323593,203.500000,203.309006,203.369995,402833
2026-07-09 19:56:00+00:00,203.104996,203.315002,203.070007,203.309998,431141
2026-07-09 19:57:00+00:00,203.151993,203.210007,203.059998,203.104996,397672
2026-07-09 19:58:00+00:00,202.840195,203.160004,202.840195,203.149994,588389
2026-07-09 19:59:00+00:00,202.729996,202.906601,202.589996,202.839996,1660427


 # Clean minute data

In [145]:
df = df.reset_index()

df = df.rename(columns={"Datetime": "Date"})

df = df[["Date", "Open", "High", "Low", "Close", "Volume"]]

df = df.copy()

df = df.dropna()

df.head()

Price,Date,Open,High,Low,Close,Volume
Ticker,,NVDA,NVDA,NVDA,NVDA,NVDA
0,2026-06-30 13:30:00+00:00,197.195007,197.240005,195.339996,195.630005,6018972
1,2026-06-30 13:31:00+00:00,195.600006,196.280594,195.110001,196.280594,817496
2,2026-06-30 13:32:00+00:00,196.289993,197.500000,196.149994,197.494995,867480
3,2026-06-30 13:33:00+00:00,197.500000,197.889999,197.214996,197.440002,869462
4,2026-06-30 13:34:00+00:00,197.440002,198.429993,197.423904,198.214996,1018742


# Download daily and options data

In [147]:
ticker_symbol = "NVDA"
ticker = yf.Ticker(ticker_symbol)

# Daily data for long-term model
daily = ticker.history(
    period="5y",
    interval="1d"
)

daily = daily.reset_index()
daily = daily[["Date", "Open", "High", "Low", "Close", "Volume"]]
daily = daily.dropna()

daily.tail()


,Date,Open,High,Low,Close,Volume
1249,2026-07-02 00:00:00-04:00,197.139999,200.059998,192.350006,194.830002,142068700
1250,2026-07-06 00:00:00-04:00,194.419998,197.550003,193.990005,195.550003,108999000
1251,2026-07-07 00:00:00-04:00,192.369995,198.410004,191.139999,196.929993,124154600
1252,2026-07-08 00:00:00-04:00,195.179993,205.160004,195.059998,204.119995,147419100
1253,2026-07-09 00:00:00-04:00,204.460007,204.589996,198.960007,202.779999,131649500


# Download options chain

In [149]:
expiration_dates = ticker.options

expiration_dates

('2026-07-10',
 '2026-07-13',
 '2026-07-15',
 '2026-07-17',
 '2026-07-20',
 '2026-07-22',
 '2026-07-24',
 '2026-07-31',
 '2026-08-07',
 '2026-08-14',
 '2026-08-21',
 '2026-09-18',
 '2026-10-16',
 '2026-11-20',
 '2026-12-18',
 '2027-01-15',
 '2027-02-19',
 '2027-03-19',
 '2027-06-17',
 '2027-09-17',
 '2027-12-17',
 '2028-01-21',
 '2028-06-16',
 '2028-12-15')

# Choose the first available expiration date

In [151]:
expiration = expiration_dates[0]

chain = ticker.option_chain(expiration)

calls = chain.calls
puts = chain.puts

calls.head()

,contractSymbol,lastTradeDate,strike,lastPrice,bid,ask,change,percentChange,volume,openInterest,impliedVolatility,inTheMoney,contractSize,currency
0,NVDA260710C00050000,2026-07-09 19:25:49+00:00,50.0,153.77,0.0,0.0,0.0,0.0,55.0,0,0.00001,True,REGULAR,USD
1,NVDA260710C00055000,2026-07-09 19:34:58+00:00,55.0,148.17,0.0,0.0,0.0,0.0,37.0,0,0.00001,True,REGULAR,USD
2,NVDA260710C00060000,2026-07-09 19:34:58+00:00,60.0,143.27,0.0,0.0,0.0,0.0,38.0,0,0.00001,True,REGULAR,USD
3,NVDA260710C00065000,2026-07-09 19:32:12+00:00,65.0,138.33,0.0,0.0,0.0,0.0,163.0,0,0.00001,True,REGULAR,USD
4,NVDA260710C00070000,2026-07-09 19:32:12+00:00,70.0,133.43,0.0,0.0,0.0,0.0,26.0,0,0.00001,True,REGULAR,USD


# Save raw datasets

In [153]:
df.to_csv("NVDA_Minute.csv", index=False)
daily.to_csv("NVDA_Daily.csv", index=False)
calls.to_csv("NVDA_Calls.csv", index=False)
puts.to_csv("NVDA_Puts.csv", index=False)

print("Saved:")
print("NVDA_Minute.csv")
print("NVDA_Daily.csv")
print("NVDA_Calls.csv")
print("NVDA_Puts.csv")

Saved:
NVDA_Minute.csv
NVDA_Daily.csv
NVDA_Calls.csv
NVDA_Puts.csv


# # STEP 7: Add indicators to minute data

In [155]:
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

df.columns

df["MA20"] = df["Close"].rolling(20).mean()
df["MA50"] = df["Close"].rolling(50).mean()

df["Return"] = df["Close"].pct_change()

df["Volume_MA20"] = df["Volume"].rolling(20).mean()
df["Relative_Volume"] = df["Volume"] / df["Volume_MA20"]

df = df.dropna()

df.tail()

Price,Date,Open,High,Low,Close,Volume,MA20,MA50,Return,Volume_MA20,Relative_Volume
2725,2026-07-09 19:55:00+00:00,203.369995,203.500000,203.309006,203.323593,402833,203.843040,203.795053,-0.000032,248829.70,1.618910
2726,2026-07-09 19:56:00+00:00,203.309998,203.315002,203.070007,203.104996,431141,203.803790,203.792755,-0.001075,259303.30,1.662690
2727,2026-07-09 19:57:00+00:00,203.104996,203.210007,203.059998,203.151993,397672,203.772889,203.791195,0.000231,271050.35,1.467152
2728,2026-07-09 19:58:00+00:00,203.149994,203.160004,202.840195,202.840195,588389,203.711149,203.783001,-0.001535,287918.65,2.043595
2729,2026-07-09 19:59:00+00:00,202.839996,202.906601,202.589996,202.729996,1660427,203.648649,203.772601,-0.000543,361055.45,4.598814


In [156]:
df.columns

Index(['Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'MA20', 'MA50',
       'Return', 'Volume_MA20', 'Relative_Volume'],
      dtype='object', name='Price')

In [157]:
df.head()

Price,Date,Open,High,Low,Close,Volume,MA20,MA50,Return,Volume_MA20,Relative_Volume
49,2026-06-30 14:19:00+00:00,197.815796,198.095001,197.779999,197.975006,435168,197.684016,197.403742,0.000784,406655.30,1.070115
50,2026-06-30 14:20:00+00:00,197.990005,197.990005,197.649994,197.729996,241817,197.701015,197.445742,-0.001238,400622.50,0.603603
51,2026-06-30 14:21:00+00:00,197.725006,197.979996,197.649994,197.860001,215852,197.721765,197.477330,0.000657,393181.10,0.548989
52,2026-06-30 14:22:00+00:00,197.839996,198.009995,197.800003,197.979904,217777,197.727261,197.487028,0.000606,388011.60,0.561264
53,2026-06-30 14:23:00+00:00,197.960007,198.130005,197.810104,198.115005,312023,197.721011,197.500528,0.000682,376651.85,0.828412


#  Add RSI to minute data

In [159]:
delta = df["Close"].diff()

gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)

avg_gain = gain.rolling(14).mean()
avg_loss = loss.rolling(14).mean()

rs = avg_gain / avg_loss

df["RSI"] = 100 - (100 / (1 + rs))

df[["Close", "RSI"]].tail()

C:\Users\Alexx\AppData\Local\Temp\ipykernel_22980\3879707744.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["RSI"] = 100 - (100 / (1 + rs))


Price,Close,RSI
2725,203.323593,27.904135
2726,203.104996,14.286388
2727,203.151993,18.908399
2728,202.840195,12.810254
2729,202.729996,12.083121


# Add MACD to minute data

In [161]:

df = df[["Date","Open","High","Low","Close","Volume"]]

ema12 = df["Close"].ewm(span=12, adjust=False).mean()
ema26 = df["Close"].ewm(span=26, adjust=False).mean()

df["MACD"] = ema12 - ema26
df["MACD_Signal"] = df["MACD"].ewm(span=9, adjust=False).mean()
df["MACD_Histogram"] = df["MACD"] - df["MACD_Signal"]

df[["MACD", "MACD_Signal", "MACD_Histogram"]].tail()

Price,MACD,MACD_Signal,MACD_Histogram
2725,-0.090112,-0.011978,-0.078134
2726,-0.130058,-0.035594,-0.094464
2727,-0.156123,-0.059700,-0.096424
2728,-0.199639,-0.087687,-0.111951
2729,-0.240248,-0.118199,-0.122048


# VWAP (Volume Weighted Average Price)

# Typical Price
df["Typical_Price"] = (
    df["High"] +
    df["Low"] +
    df["Close"]
) / 3

# Price × Volume
df["TPV"] = df["Typical_Price"] * df["Volume"]

# Running totals
df["Cumulative_TPV"] = df["TPV"].cumsum()
df["Cumulative_Volume"] = df["Volume"].cumsum()

# VWAP
df["VWAP"] = df["Cumulative_TPV"] / df["Cumulative_Volume"]

df[["Close","VWAP"]].tail()

# Bollinger Bands

In [165]:
df["BB_Middle"] = df["Close"].rolling(20).mean()

rolling_std = df["Close"].rolling(20).std()

df["BB_Upper"] = df["BB_Middle"] + (2 * rolling_std)
df["BB_Lower"] = df["BB_Middle"] - (2 * rolling_std)

df[["Close","BB_Upper","BB_Middle","BB_Lower"]].tail()

Price,Close,BB_Upper,BB_Middle,BB_Lower
2725,203.323593,204.286599,203.843040,203.399480
2726,203.104996,204.355577,203.803790,203.252002
2727,203.151993,204.397107,203.772889,203.148671
2728,202.840195,204.444310,203.711149,202.977988
2729,202.729996,204.490389,203.648649,202.806909


# Average True Range (ATR)

In [167]:
high_low = df["High"] - df["Low"]
high_close = (df["High"] - df["Close"].shift()).abs()
low_close = (df["Low"] - df["Close"].shift()).abs()

true_range = pd.concat(
    [high_low, high_close, low_close],
    axis=1
).max(axis=1)

df["ATR"] = true_range.rolling(14).mean()

df[["Close","ATR"]].tail()

Price,Close,ATR
2725,203.323593,0.172551
2726,203.104996,0.174592
2727,203.151993,0.170665
2728,202.840195,0.183515
2729,202.729996,0.192922


# Save completed minute dataset

In [169]:
minute_final = df.dropna().copy()

minute_final.to_csv(
    "NVDA_Minute_With_All_Indicators.csv",
    index=False
)

print("Minute dataset saved.")
print("Rows:", len(minute_final))
print("Columns:", len(minute_final.columns))

minute_final.tail()

Minute dataset saved.
Rows: 2662
Columns: 13


Price,Date,Open,High,Low,Close,Volume,MACD,MACD_Signal,MACD_Histogram,BB_Middle,BB_Upper,BB_Lower,ATR
2725,2026-07-09 19:55:00+00:00,203.369995,203.500000,203.309006,203.323593,402833,-0.090112,-0.011978,-0.078134,203.843040,204.286599,203.399480,0.172551
2726,2026-07-09 19:56:00+00:00,203.309998,203.315002,203.070007,203.104996,431141,-0.130058,-0.035594,-0.094464,203.803790,204.355577,203.252002,0.174592
2727,2026-07-09 19:57:00+00:00,203.104996,203.210007,203.059998,203.151993,397672,-0.156123,-0.059700,-0.096424,203.772889,204.397107,203.148671,0.170665
2728,2026-07-09 19:58:00+00:00,203.149994,203.160004,202.840195,202.840195,588389,-0.199639,-0.087687,-0.111951,203.711149,204.444310,202.977988,0.183515
2729,2026-07-09 19:59:00+00:00,202.839996,202.906601,202.589996,202.729996,1660427,-0.240248,-0.118199,-0.122048,203.648649,204.490389,202.806909,0.192922


# Daily Moving Averages

In [171]:
daily["MA20"] = daily["Close"].rolling(20).mean()
daily["MA50"] = daily["Close"].rolling(50).mean()
daily["MA100"] = daily["Close"].rolling(100).mean()
daily["MA200"] = daily["Close"].rolling(200).mean()

# Daily RSI

In [173]:
delta = daily["Close"].diff()

gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)

avg_gain = gain.rolling(14).mean()
avg_loss = loss.rolling(14).mean()

rs = avg_gain / avg_loss

daily["RSI"] = 100 - (100 / (1 + rs))

# Daily MACD

In [175]:
ema12 = daily["Close"].ewm(span=12, adjust=False).mean()
ema26 = daily["Close"].ewm(span=26, adjust=False).mean()

daily["MACD"] = ema12 - ema26
daily["MACD_Signal"] = daily["MACD"].ewm(span=9, adjust=False).mean()
daily["MACD_Histogram"] = daily["MACD"] - daily["MACD_Signal"]

# Daily Bollinger Bands

In [177]:
daily["BB_Middle"] = daily["Close"].rolling(20).mean()

std = daily["Close"].rolling(20).std()

daily["BB_Upper"] = daily["BB_Middle"] + (2 * std)
daily["BB_Lower"] = daily["BB_Middle"] - (2 * std)

# Daily ATR

In [179]:
high_low = daily["High"] - daily["Low"]
high_close = (daily["High"] - daily["Close"].shift()).abs()
low_close = (daily["Low"] - daily["Close"].shift()).abs()

true_range = pd.concat(
    [high_low, high_close, low_close],
    axis=1
).max(axis=1)

daily["ATR"] = true_range.rolling(14).mean()

# Daily Relative Volume

In [181]:
daily["Volume_MA20"] = daily["Volume"].rolling(20).mean()

daily["Relative_Volume"] = (
    daily["Volume"] /
    daily["Volume_MA20"]
)

daily = daily.dropna()

daily.tail()

,Date,Open,High,Low,Close,Volume,MA20,MA50,MA100,MA200,RSI,MACD,MACD_Signal,MACD_Histogram,BB_Middle,BB_Upper,BB_Lower,ATR,Volume_MA20,Relative_Volume
1249,2026-07-02 00:00:00-04:00,197.139999,200.059998,192.350006,194.830002,142068700,203.485000,209.796001,196.990687,191.018240,40.419855,-4.087618,-3.087175,-1.000442,203.485000,217.075494,189.894505,6.340715,157270100.0,0.903342
1250,2026-07-06 00:00:00-04:00,194.419998,197.550003,193.990005,195.550003,108999000,202.329500,209.657001,197.045890,191.121686,40.871213,-4.132244,-3.296189,-0.836055,202.329500,214.323465,190.335535,6.335714,154268940.0,0.706552
1251,2026-07-07 00:00:00-04:00,192.369995,198.410004,191.139999,196.929993,124154600,201.920999,209.602801,197.129892,191.254979,33.461205,-4.010032,-3.438957,-0.571074,201.920999,214.073137,189.768861,6.317857,149493895.0,0.830499
1252,2026-07-08 00:00:00-04:00,195.179993,205.160004,195.059998,204.119995,147419100,201.694999,209.519800,197.270694,191.394476,46.647638,-3.295022,-3.410170,0.115149,201.694999,213.483689,189.906309,6.670714,149946210.0,0.983147
1253,2026-07-09 00:00:00-04:00,204.460007,204.589996,198.960007,202.779999,131649500,201.424499,209.243200,197.429196,191.525123,48.037780,-2.804173,-3.288971,0.484798,201.424499,212.827652,190.021346,6.634999,147480560.0,0.892657


# Create prediction target

In [183]:
daily["Tomorrow_Close"] = daily["Close"].shift(-1)

daily["Target"] = (
    daily["Tomorrow_Close"] > daily["Close"]
).astype(int)

daily = daily.dropna()

daily[["Close", "Tomorrow_Close", "Target"]].tail()

,Close,Tomorrow_Close,Target
1248,197.580002,194.830002,0
1249,194.830002,195.550003,1
1250,195.550003,196.929993,1
1251,196.929993,204.119995,1
1252,204.119995,202.779999,0


# Features 

In [275]:
features = [
    "MA20",
    "MA50",
    "MA100",
    "MA200",
    "RSI",
    "MACD",
    "MACD_Signal",
    "ATR",
    "Relative_Volume"
]

X = daily[features]

y = daily["Target"]

# Train / Test Split

In [277]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    shuffle=False
)

# Logistic Regression 

In [279]:
from sklearn.linear_model import LogisticRegression

log_model = LogisticRegression(max_iter=1000)

log_model.fit(X_train, y_train)

log_predictions = log_model.predict(X_test)

# Evaluate the Model

In [281]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Accuracy:", accuracy_score(y_test, log_predictions))

print("\nClassification Report\n")
print(classification_report(y_test, log_predictions))

print("\nConfusion Matrix\n")
print(confusion_matrix(y_test, log_predictions))

Accuracy: 0.4928909952606635

Classification Report

              precision    recall  f1-score   support

           0       0.45      0.25      0.32       102
           1       0.51      0.72      0.60       109

    accuracy                           0.49       211
   macro avg       0.48      0.48      0.46       211
weighted avg       0.48      0.49      0.46       211


Confusion Matrix

[[25 77]
 [30 79]]


The Logistic Regression model achieved an overall prediction accuracy of 48.82% when forecasting the next trading day's price direction. The model demonstrated stronger performance in identifying upward market movements, with a recall of 72%, while correctly identifying downward movements only 25% of the time. These results indicate that the selected technical indicators contain some predictive information, but a linear model such as Logistic Regression is not sufficient to accurately capture the complexity of stock price movements. This serves as a baseline for comparison with more advanced machine learning models in the next phase of the project.

The confusion matrix shows that the model correctly predicted 25 down days and 78 up days, while incorrectly predicting 77 down days as up and 31 up days as down. In total, the model made 103 correct predictions out of 211 observations, producing an overall accuracy of 48.8%.

# Random Forest 

In [283]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

rf_model.fit(X_train, y_train)

rf_predictions = rf_model.predict(X_test)

# Evaluate Random Forest 

In [285]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Random Forest Accuracy:",
      accuracy_score(y_test, rf_predictions))

print("\nClassification Report\n")
print(classification_report(y_test, rf_predictions))

print("\nConfusion Matrix\n")
print(confusion_matrix(y_test, rf_predictions))

Random Forest Accuracy: 0.4976303317535545

Classification Report

              precision    recall  f1-score   support

           0       0.49      0.99      0.66       102
           1       0.80      0.04      0.07       109

    accuracy                           0.50       211
   macro avg       0.65      0.51      0.36       211
weighted avg       0.65      0.50      0.35       211


Confusion Matrix

[[101   1]
 [105   4]]


The Random Forest model achieved an overall accuracy of 48.82%, matching the Logistic Regression model. However, the confusion matrix shows a different prediction pattern. The model correctly identified 95 downward trading days but only 8 upward trading days. It incorrectly classified 7 downward days as upward and 101 upward days as downward. This indicates that the Random Forest model strongly favored predicting downward market movements, resulting in very high recall (93%) for down days but very low recall (7%) for up days. Although the overall accuracy remained similar to Logistic Regression, the model exhibited a strong prediction bias toward one class and did not provide balanced performance.

Confusion Matrix Explained
Actual \ Predicted	Down	Up
Actual Down	95	7
Actual Up	101	8
95 = Correctly predicted Down (True Negatives)
7 = Predicted Up, but the market actually went Down (False Positives)
101 = Predicted Down, but the market actually went Up (False Negatives)
8 = Correctly predicted Up (True Positives)

In [231]:
pip install xgboost

In [287]:
from xgboost import XGBClassifier

# TrainXGBoost

In [289]:
xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    random_state=42,
    eval_metric="logloss"
)

xgb_model.fit(X_train, y_train)

xgb_predictions = xgb_model.predict(X_test)

# Evaluate XGBoost 

In [293]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("XGBoost Accuracy:",
      accuracy_score(y_test, xgb_predictions))

print("\nClassification Report\n")
print(classification_report(y_test, xgb_predictions))

print("\nConfusion Matrix\n")
print(confusion_matrix(y_test, xgb_predictions))

XGBoost Accuracy: 0.4881516587677725

Classification Report

              precision    recall  f1-score   support

           0       0.48      0.61      0.53       102
           1       0.51      0.38      0.43       109

    accuracy                           0.49       211
   macro avg       0.49      0.49      0.48       211
weighted avg       0.49      0.49      0.48       211


Confusion Matrix

[[62 40]
 [68 41]]


The XGBoost model achieved an overall accuracy of 48.34%, which was similar to the Logistic Regression and Random Forest models. The confusion matrix showed that the model correctly predicted 69 downward trading days and 33 upward trading days, while incorrectly classifying 33 downward days as upward and 76 upward days as downward. Overall, the model produced a more balanced distribution of predictions than the Random Forest model but did not improve overall predictive accuracy.

Model Comparison
Model	Accuracy
Logistic Regression	48.82%
Random Forest	48.82%
XGBoost	48.34%

# Feature Importance

In [301]:
importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": xgb_model.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

print(importance)

           Feature  Importance
5             MACD    0.127154
1             MA50    0.124836
6      MACD_Signal    0.122024
8  Relative_Volume    0.114408
2            MA100    0.110317
4              RSI    0.107866
7              ATR    0.104789
3            MA200    0.094752
0             MA20    0.093853


The feature importance analysis indicates that MACD, Relative Volume, and the 50-day Moving Average were the most influential variables in the XGBoost model. Indicators related to momentum and trading activity contributed more to the model's predictions than longer-term trend indicators such as the 200-day Moving Average. Although the overall predictive accuracy remained modest, the feature importance results provide useful insight into which technical indicators had the greatest influence on the model's decision-making process.

# Lag Features

In [299]:
daily["Return"] = daily["Close"].pct_change()

print(daily[["Close", "Return"]].head())

daily["Lag_Return"] = daily["Return"].shift(1)

daily["Lag_RSI"] = daily["RSI"].shift(1)

daily["Lag_MACD"] = daily["MACD"].shift(1)

daily["Lag_Volume"] = daily["Relative_Volume"].shift(1)

daily["Lag_ATR"] = daily["ATR"].shift(1)

daily = daily.dropna()

daily.tail()

         Close    Return
203  18.513296       NaN
204  19.497501  0.053162
205  19.566372  0.003532
206  20.297047  0.037343
207  18.809752 -0.073276


,Date,Open,High,Low,Close,Volume,MA20,MA50,MA100,MA200,...,Volume_MA20,Relative_Volume,Tomorrow_Close,Target,Return,Lag_Return,Lag_RSI,Lag_MACD,Lag_Volume,Lag_ATR
1248,2026-07-01 00:00:00-04:00,196.199997,199.850006,193.449997,197.580002,146147600,204.481000,209.897001,196.896386,190.932742,...,158212015.0,0.923745,194.830002,0,-0.012544,0.026260,43.177220,-3.879038,1.036768,6.343573
1249,2026-07-02 00:00:00-04:00,197.139999,200.059998,192.350006,194.830002,142068700,203.485000,209.796001,196.990687,191.018240,...,157270100.0,0.903342,195.550003,1,-0.013918,-0.012544,47.375235,-3.897893,0.923745,6.227145
1250,2026-07-06 00:00:00-04:00,194.419998,197.550003,193.990005,195.550003,108999000,202.329500,209.657001,197.045890,191.121686,...,154268940.0,0.706552,196.929993,1,0.003696,-0.013918,40.419855,-4.087618,0.903342,6.340715
1251,2026-07-07 00:00:00-04:00,192.369995,198.410004,191.139999,196.929993,124154600,201.920999,209.602801,197.129892,191.254979,...,149493895.0,0.830499,204.119995,1,0.007057,0.003696,40.871213,-4.132244,0.706552,6.335714
1252,2026-07-08 00:00:00-04:00,195.179993,205.160004,195.059998,204.119995,147419100,201.694999,209.519800,197.270694,191.394476,...,149946210.0,0.983147,202.779999,0,0.036510,0.007057,33.461205,-4.010032,0.830499,6.317857


# completed Python daily dataset

In [304]:
daily.to_csv("NVDA_Daily_For_R.csv", index=False)

print("Saved NVDA_Daily_For_R.csv")

Saved NVDA_Daily_For_R.csv
